# 2. Boiling point: Chemprop graph neural network

    **Demo purpose:** a compact “gold-standard” molecular-regression workflow. Chemprop converts each SMILES string to a molecular graph, passes messages along bonds, pools the learned atom-level representation, and predicts boiling point.

    This uses a random split for a fast workshop demonstration. A scaffold split is a better follow-up when the scientific claim is generalization to new chemical families.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

SEED = 42
SAMPLE_SIZE = 1500       # small enough for a workshop CPU run
EPOCHS = 20
ACQUIRED_DATA_FILE = Path('outputs/nist_acquisition/NIST_BP_SMILES.parquet')
PACKAGED_DATA_FILE = Path('data/CompoundData.parquet')
DATA_FILE = ACQUIRED_DATA_FILE if ACQUIRED_DATA_FILE.exists() else PACKAGED_DATA_FILE
OUTPUT = Path('outputs/bp_chemprop')
OUTPUT.mkdir(parents=True, exist_ok=True)

data = pd.read_parquet(DATA_FILE).dropna(subset=['SMILES', 'BP']).drop_duplicates('SMILES')
print(f'Using data source: {DATA_FILE}')
data = data[['SMILES', 'BP']].sample(n=min(SAMPLE_SIZE, len(data)), random_state=SEED).reset_index(drop=True)

indices = np.arange(len(data))
train, held_out = train_test_split(indices, test_size=0.20, random_state=SEED)
val, test = train_test_split(held_out, test_size=0.50, random_state=SEED)
data['split'] = 'train'
data.loc[val, 'split'] = 'val'
data.loc[test, 'split'] = 'test'

csv_file = OUTPUT / 'bp_data.csv'
data.to_csv(csv_file, index=False)
display(data.groupby('split')['BP'].agg(['size', 'mean', 'min', 'max']).round(2))


In [ ]:
def chemprop_executable():
    executable = shutil.which('chemprop')
    if executable:
        return executable
    windows_candidate = Path(sys.executable).parent / 'Scripts' / 'chemprop.exe'
    if windows_candidate.exists():
        return str(windows_candidate)
    raise RuntimeError('Chemprop was not found. Create the workshop environment from environment.yml and select its kernel.')

model_dir = OUTPUT / 'model'
command = [
    chemprop_executable(), 'train', '-q', '-q',
    '--data-path', str(csv_file), '--smiles-columns', 'SMILES', '--target-columns', 'BP',
    '--task-type', 'regression', '--splits-column', 'split', '--output-dir', str(model_dir),
    '--epochs', str(EPOCHS), '--batch-size', '64', '--depth', '3', '--dropout', '0.05',
    '--pytorch-seed', str(SEED), '--num-workers', '0',
]
print(subprocess.list2cmdline(command))
subprocess.run(command, check=True)

prediction_file = OUTPUT / 'predictions.csv'
predict_command = [
    chemprop_executable(), 'predict', '--test-path', str(csv_file), '--smiles-columns', 'SMILES',
    '--model-paths', str(model_dir), '--preds-path', str(prediction_file),
    '--batch-size', '64', '--num-workers', '0',
]
subprocess.run(predict_command, check=True)


In [ ]:
predictions = pd.read_csv(prediction_file)
candidate_columns = ['BP_pred', 'pred_0', 'BP']
prediction_column = next((column for column in candidate_columns if column in predictions.columns), None)
if prediction_column is None:
    numeric = predictions.select_dtypes(include='number').columns.tolist()
    if len(numeric) != 1:
        raise ValueError(f'Cannot identify prediction column: {predictions.columns.tolist()}')
    prediction_column = numeric[0]

results = data.copy()
results['predicted_BP'] = predictions[prediction_column].to_numpy()
rows = []
for split, subset in results.groupby('split', sort=False):
    rows.append({
        'split': split, 'n': len(subset),
        'MAE (K)': mean_absolute_error(subset.BP, subset.predicted_BP),
        'RMSE (K)': mean_squared_error(subset.BP, subset.predicted_BP) ** 0.5,
        'R²': r2_score(subset.BP, subset.predicted_BP),
    })
metrics = pd.DataFrame(rows).set_index('split')
display(metrics.round(2))

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
limits = [min(results.BP.min(), results.predicted_BP.min()), max(results.BP.max(), results.predicted_BP.max())]
for ax, split in zip(axes, ['train', 'val', 'test']):
    subset = results.query('split == @split')
    ax.scatter(subset.BP, subset.predicted_BP, s=18, alpha=0.65)
    ax.plot(limits, limits, 'k--')
    ax.set(title=f'{split.title()} (n={len(subset)})', xlabel='Measured BP (K)')
    ax.grid(alpha=0.2)
axes[0].set_ylabel('Predicted BP (K)')
fig.suptitle('Chemprop boiling-point parity')
fig.tight_layout()
plt.show()


## Discussion

    The test panel is the relevant fast demonstration result; the validation partition selected the training epoch implicitly through Chemprop’s training process. The next notebook asks a complementary question: what can a pretrained molecular language model contribute before task-specific neural-network training?
